# 03 — Noise-Induced Transitions and Stochastic Triggering

This notebook turns the spotlight onto **noise as a dynamical mechanism**.  
Its purpose is not only to simulate reversals, but to understand how stochastic forcing changes the qualitative regime of a reduced model.

## Learning goals

- understand the role of noise amplitude in metastable switching,
- compare weak-noise and strong-noise regimes,
- analyze transition statistics across ensembles,
- connect stochastic forcing to unresolved geodynamo variability.


In [ ]:

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10, 4.5)
plt.rcParams["axes.grid"] = True

ROOT = Path.cwd()
if not (ROOT / "models").exists() and (ROOT.parent / "models").exists():
    ROOT = ROOT.parent

print("Working directory:", Path.cwd())
print("Repository root guessed as:", ROOT)

def repo_file_info(relpath):
    p = ROOT / relpath
    return {"exists": p.exists(), "size": p.stat().st_size if p.exists() else None, "path": str(p)}

def sign_changes(x):
    s = np.sign(x)
    s[s == 0] = np.nan
    valid = ~np.isnan(s)
    sv = s[valid]
    return int(np.sum(sv[1:] * sv[:-1] < 0))

def polarity_series(x):
    p = np.sign(x)
    if len(p) == 0:
        return p
    # carry last sign through zeros
    for i in range(1, len(p)):
        if p[i] == 0:
            p[i] = p[i-1]
    if p[0] == 0:
        nz = np.flatnonzero(p != 0)
        if len(nz):
            p[:nz[0]] = p[nz[0]]
    return p

def residence_times_from_signal(x, dt=1.0):
    p = polarity_series(np.asarray(x))
    if len(p) == 0:
        return np.array([])
    durations = []
    current = p[0]
    count = 1
    for val in p[1:]:
        if val == current:
            count += 1
        else:
            durations.append(count * dt)
            current = val
            count = 1
    durations.append(count * dt)
    return np.array(durations)

def power_spectrum(x, dt=1.0):
    x = np.asarray(x)
    x = x - np.mean(x)
    freqs = np.fft.rfftfreq(len(x), d=dt)
    spec = np.abs(np.fft.rfft(x))**2 / len(x)
    return freqs[1:], spec[1:]

rng = np.random.default_rng(42)


## 1. Why noise matters

In reduced geomagnetic modeling, the term “noise” does not usually mean literal thermal noise.  
Instead, it represents unresolved fluctuations from a much richer hidden system.

We again use the stochastic bistable equation

$$
dx = (ax - bx^3)\,dt + \sigma\,dW_t,
$$

but now the main control parameter is \(\sigma\).


In [ ]:

def simulate_bistable_once(sigma, n_steps=30000, dt=0.01, a=1.0, b=1.0, seed=42):
    rng = np.random.default_rng(seed)
    x = np.zeros(n_steps)
    for i in range(1, n_steps):
        drift = a * x[i-1] - b * x[i-1]**3
        x[i] = x[i-1] + drift * dt + sigma * np.sqrt(dt) * rng.standard_normal()
    return np.arange(n_steps) * dt, x

sigmas = [0.15, 0.35, 0.60, 0.90]
fig, axes = plt.subplots(len(sigmas), 1, figsize=(10, 10), sharex=True)

for ax, s in zip(axes, sigmas):
    t, x = simulate_bistable_once(s, seed=42)
    ax.plot(t, x, lw=0.7)
    ax.set_ylabel(f"s={s}")

axes[0].set_title("Noise-induced regime changes")
axes[-1].set_xlabel("time")
plt.tight_layout()
plt.show()


## 2. Ensemble statistics

A robust notebook should move beyond single realizations.  
We now compute reversal counts across an ensemble for each noise level.


In [ ]:

def estimate_reversals(x, dt=0.01, persistence=20):
    return len(reversal_times(x, dt=dt, persistence=persistence))

sigmas = np.linspace(0.15, 1.0, 10)
ensemble_size = 20
mean_counts, std_counts = [], []

for s in sigmas:
    counts = []
    for k in range(ensemble_size):
        _, x = simulate_bistable_once(float(s), seed=100 + k)
        counts.append(estimate_reversals(x))
    mean_counts.append(np.mean(counts))
    std_counts.append(np.std(counts))

fig, ax = plt.subplots()
ax.errorbar(sigmas, mean_counts, yerr=std_counts, marker="o", capsize=3)
ax.set_title("Ensemble reversal statistics versus noise amplitude")
ax.set_xlabel("sigma")
ax.set_ylabel("mean reversal count ± std")
plt.show()


## 3. Occupancy statistics

Another useful diagnostic is the fraction of time spent in each polarity state.


In [ ]:

def occupancy_fraction(x):
    p = polarity_series(x)
    pos = np.mean(p > 0)
    neg = np.mean(p < 0)
    return pos, neg

fractions = []
for s in sigmas:
    _, x = simulate_bistable_once(float(s), seed=42)
    fractions.append(occupancy_fraction(x))

fractions = np.array(fractions)
fig, ax = plt.subplots()
ax.plot(sigmas, fractions[:,0], marker="o", label="positive occupancy")
ax.plot(sigmas, fractions[:,1], marker="o", label="negative occupancy")
ax.set_title("Occupancy fractions")
ax.set_xlabel("sigma")
ax.set_ylabel("fraction of time")
ax.legend()
plt.show()


## 4. Spectral viewpoint

Noise also changes the spectral character of the signal.  
Below we compare spectra for weak and strong forcing.


In [ ]:

fig, ax = plt.subplots()
for s in [0.2, 0.6, 1.0]:
    _, x = simulate_bistable_once(s, seed=42)
    f, S = power_spectrum(x, dt=0.01)
    ax.loglog(f, S, label=f"sigma={s}")

ax.set_title("Power spectra under different stochastic forcing")
ax.set_xlabel("frequency")
ax.set_ylabel("power")
ax.legend()
plt.show()


## 5. Interpretation

The main lesson is that the **same deterministic landscape** can produce very different reversal statistics depending on the amplitude of unresolved forcing.

This is precisely why stochastic terms are scientifically important in reduced geomagnetic models.


## 6. Connection to the repository

This notebook is most closely aligned with:

- `models/bistable_models/stochastic_forcing.py`
- `diagnostics/reversal_statistics.py`
- `diagnostics/power_spectra.py`

It can later be upgraded to import repository functions directly once those scripts contain callable code.


## 7. Suggested exercises

1. Increase the ensemble size and check convergence of the mean reversal count.  
2. Study how the result changes with `dt`.  
3. Replace Gaussian noise with colored noise and compare the outcome.  
4. Compute the waiting-time distribution for two values of `sigma`.
